# CRC preprocessing

This notebook builds the local `preprocessed_crc_data.pkl` file required by `evaluation/main_CRC_censoring.py`.

Expected private input:

```text
data/a_endpt.sas7bdat
```

Generated local output:

```text
data/use_cases/preprocessed_crc_data.pkl
```


In [ ]:
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / 'evaluation').exists() and (path / 'data').exists():
            return path
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def decode_series_values(series: pd.Series) -> pd.Series:
    if series.dtype != object:
        return series
    return series.apply(lambda value: value.decode('utf-8') if isinstance(value, (bytes, bytearray)) else value)


repo_root = find_repo_root(Path.cwd())
raw_crc_path = repo_root / 'data' / 'a_endpt.sas7bdat'
crc_output_dir = repo_root / 'data' / 'use_cases'
crc_output_dir.mkdir(parents=True, exist_ok=True)
output_path = crc_output_dir / 'preprocessed_crc_data.pkl'

if not raw_crc_path.exists():
    raise FileNotFoundError(f'Missing CRC source file: {raw_crc_path}')

raw_df = pd.read_sas(raw_crc_path, format='sas7bdat')
raw_df.columns = [column.decode('utf-8') if isinstance(column, (bytes, bytearray)) else str(column) for column in raw_df.columns]
for column in raw_df.columns:
    raw_df[column] = decode_series_values(raw_df[column])

alias_groups = {
    'AGE': ['AGE'],
    'SEX': ['SEX'],
    'RACCAT': ['RACCAT'],
    'PRBEV': ['PRBEV'],
    'PROXAL': ['PROXAL'],
    'B_METACT': ['B_METACT'],
    'B_LDHNM': ['B_LDHNM'],
    'BECOGICD': ['BECOGICD'],
    'DIAGTYPE': ['DIAGTYPE'],
    'PRADJYN': ['PRADJYN'],
    'LIVERMET': ['LIVERMET'],
    'KRAS': ['KRAS'],
    'time': ['TIME', 'OS', 'OS_TIME', 'SURVIVAL_TIME'],
    'event': ['EVENT', 'STATUS', 'OS_EVENT'],
    'tratamiento': ['TRATAMIENTO', 'TREATMENT', 'TRT', 'ARM'],
}

column_lookup = {str(column).upper(): column for column in raw_df.columns}
rename_map = {}
missing_columns = []
for target, aliases in alias_groups.items():
    matched_column = None
    for alias in aliases:
        if alias in column_lookup:
            matched_column = column_lookup[alias]
            break
    if matched_column is None:
        missing_columns.append(target)
    else:
        rename_map[matched_column] = target

if missing_columns:
    raise KeyError(f'The CRC source file is missing the columns required by this pipeline: {missing_columns}')

data_df = raw_df.rename(columns=rename_map)[list(alias_groups.keys())].copy()
data_df = data_df.apply(pd.to_numeric, errors='coerce')
data_df = data_df.dropna(subset=['time', 'event', 'tratamiento']).reset_index(drop=True)

print(f'Repository root: {repo_root}')
print(f'Loaded CRC source data with shape: {data_df.shape}')
print(data_df.head())


In [ ]:
# Make the data observational by introducing treatment-selection bias.
data_t0 = data_df[data_df['tratamiento'] == 0]
data_t1 = data_df[data_df['tratamiento'] == 1]
ate_ground_truth = data_t1['time'].mean() - data_t0['time'].mean()

age_threshold = 50
d0_young = data_t0[data_t0['AGE'] <= age_threshold]
d0_old = data_t0[data_t0['AGE'] > age_threshold]
d1_young = data_t1[data_t1['AGE'] <= age_threshold]
d1_old = data_t1[data_t1['AGE'] > age_threshold]

d0_young = d0_young.sample(frac=0.8, random_state=0)
d0_old = d0_old.sample(frac=0.2, random_state=0)
d1_young = d1_young.sample(frac=0.2, random_state=0)
d1_old = d1_old.sample(frac=0.8, random_state=0)

d1 = pd.concat([d1_young, d1_old], axis=0).reset_index(drop=True)
d0 = pd.concat([d0_young, d0_old], axis=0).reset_index(drop=True)

ate_obs = d1['time'].mean() - d0['time'].mean()
data_raw = pd.concat([d0, d1], axis=0).sample(frac=1, random_state=0).reset_index(drop=True)

print(f'Ground truth ATE: {ate_ground_truth:.4f}')
print(f'Observed ATE after biasing the treatment selection: {ate_obs:.4f}')


In [ ]:
# Data preprocessing
y_df = data_raw['time'] + 1.0
t_df = data_raw['tratamiento']
c_df = 1.0 - data_raw['event']
X_df = data_raw.drop(columns=['time', 'tratamiento', 'event'])
mask = data_raw.isnull().astype(int)
mask = mask[X_df.columns].values

feature_types = []
for column in X_df.columns:
    unique_values = X_df[column].dropna().unique()
    if len(unique_values) == 2:
        feature_types.append('binary')
    elif len(unique_values) < 10:
        feature_types.append('categorical')
    else:
        feature_types.append('continuous')

feature_types = pd.DataFrame({'features': X_df.columns, 'types': feature_types})
for index in range(len(feature_types)):
    if feature_types.loc[index, 'types'] == 'categorical':
        feature = feature_types.loc[index, 'features']
        num_categories = X_df[feature].nunique()
        feature_types.loc[index, 'types'] = f'categorical_{num_categories}'
        X_df[feature] = pd.Categorical(X_df[feature]).codes

continuous_vars = feature_types[feature_types['types'] == 'continuous']['features'].values
data_types = [feature_types[feature_types['features'] == column]['types'].values[0] for column in X_df.columns]
num_treatments = len(np.unique(t_df.to_numpy()))


In [ ]:
# Hold out a test set used by the downstream evaluation script.
(X_train_df, X_test_df,
 y_train_df, y_test_df,
 t_train_df, t_test_df,
 c_train_df, c_test_df,
 mask_train_, mask_test) = train_test_split(
    X_df, y_df, t_df, c_df, mask, test_size=0.2, random_state=0
)

X_train_val = X_train_df.values
X_test = X_test_df.values
t_test = t_test_df.values
y_test = y_test_df.values
c_test = c_test_df.values

for index, column in enumerate(X_train_df.columns):
    if column in continuous_vars:
        mean = np.mean(X_train_val[:, index])
        std = np.std(X_train_val[:, index])
        if std > 0.0:
            X_train_val[:, index] = (X_train_val[:, index] - mean) / std
            X_test[:, index] = (X_test[:, index] - mean) / std


In [ ]:
def change_censoring(X_array, y_array, c_array, setting):
    if setting == 'no_censor':
        return y_array.copy(), c_array.copy()

    if setting != 'ic':
        raise ValueError(f'Unknown setting: {setting}')

    rng = np.random.default_rng(seed=42)
    age_normalized = (X_array[:, 0] - X_array[:, 0].min()) / (X_array[:, 0].max() - X_array[:, 0].min())
    age_normalized = age_normalized * 0.8 + 0.1
    time_mean_per_patient = 5 * y_array.mean() * age_normalized
    time_to_censoring = rng.exponential(scale=time_mean_per_patient)

    new_c_array = np.zeros_like(c_array)
    new_y_array = y_array.copy()
    newly_censored = time_to_censoring < y_array

    new_c_array[newly_censored] = 1
    new_y_array[newly_censored] = time_to_censoring[newly_censored]

    already_censored = c_array == 1
    new_c_array[already_censored] = 1
    new_y_array[already_censored] = y_array[already_censored]
    new_y_array[new_c_array == 1] = np.maximum(new_y_array[new_c_array == 1], 1)

    return new_y_array, new_c_array


In [ ]:
data_to_save = {}
n_bootstraps = 100
settings = ['no_censor', 'ic']

for setting in settings:
    data_to_save[setting] = {}
    y_setting, c_setting = change_censoring(X_train_val.copy(), y_train_df.values.copy(), c_train_df.values.copy(), setting)
    y_setting_test, c_setting_test = change_censoring(X_test.copy(), y_test.copy(), c_test.copy(), setting)

    for seed in range(n_bootstraps):
        (X_train, X_val,
         y_train, y_val,
         t_train, t_val,
         c_train, c_val,
         mask_train, mask_val) = train_test_split(
            X_train_val, y_setting, t_train_df.values, c_setting, mask_train_, test_size=0.2, random_state=seed
        )

        data = {
            'X_train': X_train, 't_train': t_train, 'y_train': y_train, 'c_train': c_train, 'mask_train': mask_train,
            'X_val': X_val, 't_val': t_val, 'y_val': y_val, 'c_val': c_val, 'mask_val': mask_val,
            'X_test': X_test, 't_test': t_test, 'y_test': y_setting_test, 'c_test': c_setting_test, 'mask_test': mask_test,
            'data_types': data_types, 'input_dim': X_train_df.shape[1], 'num_treatments': num_treatments,
            'num_val_data': X_val.shape[0], 'x_columns': X_train_df.columns.tolist(),
            'ate_ground_truth': ate_ground_truth, 'ate_obs': ate_obs,
        }
        data_to_save[setting][str(seed)] = data

with open(output_path, 'wb') as file_handle:
    pickle.dump(data_to_save, file_handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved CRC preprocessing output to: {output_path}')


In [ ]:
with open(output_path, 'rb') as file_handle:
    loaded_data = pickle.load(file_handle)

for setting in settings:
    print(setting, loaded_data[setting]['0']['X_train'].shape, loaded_data[setting]['0']['X_test'].shape)
